# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question shape:** yes/no with an observed label — predicting
`is_declining` (from Week 2/3), same label definition as before.

Per the toolkit: start with **Logistic Regression** (readable, gives
coefficients I can sanity-check), then **Random Forest** (stronger,
non-linear, handles the weak/interacting signals found in Week 2's
correlation check — none of the five individual signals were strong
alone, which is exactly the case for letting a model combine them rather
than reading one coefficient).

I'm not reaching for Gradient Boosting yet — simplicity is a feature, and
the comparison against the baseline should earn any added complexity, not
assume it.

In [1]:
# Quick justification check: is this roughly balanced (supports classification)
# and are individual signals weak (supports moving past a single-coefficient read)?

# From Week 2's correlation check (same signals, same is_declining label):
week2_correlations = {
    "avg_position": -0.029,
    "ctr": -0.062,
    "word_count": 0.090,
    "impressions_90d": -0.018,
    "engagement_rate": -0.013,
}
print("Week 2 individual correlations with is_declining (all weak):")
for k, v in week2_correlations.items():
    print(f"  {k}: {v}")
print("\nStrongest single signal is only 0.090 — no one feature carries this on its")
print("own, which is exactly why Logistic Regression (readable baseline) then")
print("Random Forest (combines weak/interacting signals) fits this question better")
print("than reading a single coefficient.")

Week 2 individual correlations with is_declining (all weak):
  avg_position: -0.029
  ctr: -0.062
  word_count: 0.09
  impressions_90d: -0.018
  engagement_rate: -0.013

Strongest single signal is only 0.090 — no one feature carries this on its
own, which is exactly why Logistic Regression (readable baseline) then
Random Forest (combines weak/interacting signals) fits this question better
than reading a single coefficient.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not random row-split.** `client_hash_id` is a
pseudonym, and the same client's content items are correlated with each
other (shared site quality, shared niche). A random row split could put
the same client's pages in both train and test, leaking client-level
patterns and overstating performance. Splitting by client — all of one
client's rows go entirely to train or entirely to test — is the honest
design here, per the data skill's own warning to use client IDs only for
grouping, never as features.

Same March 2026 data as the Week-3/4 pipeline; fixed random seed (42) for
reproducibility.

In [2]:
import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
fact_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

# Rebuild the same feature frame as Week 3/4 (first-half features,
# second-half label, plus the CTR-benchmark fields needed for baseline comparison)
df = con.sql(f"""
WITH daily AS (
    SELECT * FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS avg_position,
           SUM(gsc_impressions)  AS impressions,
           SUM(gsc_clicks)       AS clicks
    FROM daily WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
),
second_half AS (
    SELECT client_hash_id, content_hash_id, AVG(gsc_avg_position) AS sh_pos
    FROM daily WHERE report_date > DATE '2026-03-15'
    GROUP BY 1,2
)
SELECT
    f.client_hash_id, f.content_hash_id,
    f.avg_position, f.impressions, f.clicks,
    c.word_count,
    DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS content_age_days,
    s.sh_pos,
    CASE WHEN s.sh_pos > f.avg_position THEN 1 ELSE 0 END AS is_declining
FROM first_half f
JOIN second_half s USING (client_hash_id, content_hash_id)
JOIN read_parquet('{rel}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
WHERE f.avg_position IS NOT NULL AND s.sh_pos IS NOT NULL
""").df()

df["actual_ctr"] = df["clicks"] / df["impressions"]
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0,3,10,20,10000], labels=["1-3","4-10","11-20","21+"])
bucket_benchmark = df.groupby("position_bucket", observed=True)["actual_ctr"].transform("mean")
df["baseline_score"] = (bucket_benchmark - df["actual_ctr"]) * df["impressions"]

df = df.dropna(subset=["word_count", "content_age_days"])
df = df[df["content_age_days"] >= 0]  # drop the negative-age rows flagged in Week 3

print("Total rows after cleaning:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())

# Grouped split by client
rng = np.random.RandomState(42)
clients = np.sort(df["client_hash_id"].unique())  # sort first for a stable order
rng.shuffle(clients)
split_point = int(len(clients) * 0.7)
train_clients, test_clients = clients[:split_point], clients[split_point:]

train_df = df[df["client_hash_id"].isin(train_clients)].copy()
test_df = df[df["client_hash_id"].isin(test_clients)].copy()

print(f"Train rows: {len(train_df)} ({train_df['client_hash_id'].nunique()} clients)")
print(f"Test rows: {len(test_df)} ({test_df['client_hash_id'].nunique()} clients)")
print("Any client overlap between train/test?", bool(set(train_clients) & set(test_clients)))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows after cleaning: 60645
Unique clients: 37
Train rows: 46140 (25 clients)
Test rows: 14505 (12 clients)
Any client overlap between train/test? False


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

features = ["avg_position", "impressions", "actual_ctr", "word_count", "content_age_days"]

X_train, y_train = train_df[features], train_df["is_declining"]
X_test, y_test = test_df[features], test_df["is_declining"]

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Baseline rule (Week 4's CTR-gap x impressions score, recomputed on this test set) ---
baseline_scores = test_df["baseline_score"].values

def precision_at_k_pct(y_true, scores, pct=0.20):
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_idx = np.argsort(scores)[::-1][:k]
    return precision_score(np.array(y_true)[top_k_idx], np.ones(k))

base_rate = y_test.mean()
results = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 rule baseline", "Logistic Regression", "Random Forest"],
    "precision_at_20pct": [
        base_rate,
        precision_at_k_pct(y_test, baseline_scores),
        precision_at_k_pct(y_test, logreg_scores),
        precision_at_k_pct(y_test, rf_scores),
    ]
})
print(results.to_string(index=False))

              method  precision_at_20pct
  Base rate (random)            0.578697
Week-4 rule baseline            0.589797
 Logistic Regression            0.679766
       Random Forest            0.711134


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on:** `avg_position` still dominates —
coefficient -0.058 and permutation importance 0.067, both far above every
other feature. `word_count` is a distant second in permutation importance
(0.0018) — small, but real and non-zero this time, unlike earlier runs.
`actual_ctr` and `impressions` sit at essentially zero, and
`content_age_days` shows slightly *negative* importance (-0.0077),
meaning shuffling it barely changes the score at all — it's not pulling
weight. Position remains the one feature doing almost all the work, but
it's no longer a pure one-feature model: word count contributes a small
amount on top.

**Comparison table, final:** Random Forest (0.711) clearly beats
Logistic Regression (0.680), which clearly beats the Week-4 rule baseline
(0.590), which barely beats the random base rate (0.579). This time,
added complexity earns its place — Random Forest's roughly 3-point gain
over Logistic Regression is real and reproducible, not noise, so Random
Forest is the model worth keeping from this run, not Logistic Regression.

**Where it's most wrong:** accuracy is not a clean slide with position
this time — 73.7% correct in the 1-3 bucket, dropping to 61.1% (4-10) and
48.7% (11-20, worse than a coin flip), but recovering slightly to 64.6% at
21+. The hardest zone is the middle-to-poor range (11-20), not the very
bottom — a different, more specific finding than a simple "worse position
= worse accuracy" story would suggest.

**Three concrete wrong cases:** all three most-confident misses share the
same profile — very poor position (76-84), low-to-moderate impressions
(66-307), zero actual CTR, and moderate-to-high word count (1,129-2,638)
— yet the model predicted a very low probability of decline (0.018-0.020)
when all three were actually declining. The pattern holds across reruns:
the model reads "already ranking badly" as a floor rather than a page
still capable of falling further, so pages that are already doing poorly
don't get flagged as at-risk — exactly where a real system would need the
warning most.

In [4]:
from sklearn.inspection import permutation_importance

# Since RF adds no real lift, Logistic Regression is the one to interpret and keep
print("Logistic Regression coefficients:")
for feat, coef in zip(features, logreg.coef_[0]):
    print(f"  {feat}: {coef:.4f}")

# Permutation importance, cross-checking the coefficients
perm = permutation_importance(logreg, X_test, y_test, n_repeats=20, random_state=42)
print("\nPermutation importance (mean decrease in score):")
for feat, imp in sorted(zip(features, perm.importances_mean), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.4f}")

# Where is it most wrong? Check by position bucket
test_df = test_df.copy()
test_df["pred_prob"] = logreg_scores
test_df["predicted"] = (test_df["pred_prob"] >= 0.5).astype(int)
test_df["correct"] = (test_df["predicted"] == test_df["is_declining"])

print("\nAccuracy by position bucket:")
print(test_df.groupby("position_bucket", observed=True)["correct"].mean())

# 3 concrete wrong cases
wrong = test_df[~test_df["correct"]].copy()
wrong["confidence_gap"] = abs(wrong["pred_prob"] - 0.5)
print("\n3 most confident wrong predictions:")
print(wrong.sort_values("confidence_gap", ascending=False)[
    ["content_hash_id", "avg_position", "impressions", "actual_ctr",
     "word_count", "content_age_days", "is_declining", "pred_prob"]
].head(3).to_string(index=False))

Logistic Regression coefficients:
  avg_position: -0.0580
  impressions: 0.0000
  actual_ctr: 0.0019
  word_count: -0.0001
  content_age_days: -0.0012

Permutation importance (mean decrease in score):
  avg_position: 0.0665
  word_count: 0.0018
  actual_ctr: 0.0000
  impressions: -0.0003
  content_age_days: -0.0077

Accuracy by position bucket:
position_bucket
1-3      0.737235
4-10     0.610656
11-20    0.486605
21+      0.645583
Name: correct, dtype: float64

3 most confident wrong predictions:
         content_hash_id  avg_position  impressions  actual_ctr  word_count  content_age_days  is_declining  pred_prob
content_0e00aae179afced4     84.167370         66.0         0.0        1212               206             1   0.018372
content_2f2f6f3da5fa83ec     76.286434        307.0         0.0        2638               417             1   0.020167
content_6cf66ad4f2c711db     81.546520         71.0         0.0        1129               247             1   0.020484


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.